# RAG系统构建全流程详解

## 一、RAG系统概述

检索增强生成(Retrieval-Augmented Generation，RAG)是当前最先进的问答系统架构之一，它通过结合信息检索与大型语言模型，有效解决了传统生成式AI的幻觉问题。RAG系统的核心思想是：先检索相关知识，再基于检索结果生成回答。

### RAG系统的核心优势

1. **知识实时性**：通过检索外部知识源，不受限于模型训练数据
2. **回答可靠性**：基于实际文档生成回答，减少幻觉现象
3. **可解释性**：可提供回答所依据的参考文档
4. **模块化设计**：各组件可独立优化升级

## 二、RAG系统核心组件

### 1. 文档加载(Loader)

文档加载是RAG系统的第一步，负责从各种来源获取原始文本数据。常见的文档加载器包括：

- **PDF加载器**：处理学术论文、技术手册等PDF文档
- **网页加载器**：抓取网页内容作为知识来源
- **文本加载器**：读取纯文本文件
- **数据库加载器**：从SQL/NoSQL数据库提取数据

选择加载器时需要考虑：
- 文档格式兼容性
- 编码处理能力(特别是中文文档)
- 元数据保留需求

#### 1.1 PDF加载器

In [3]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "./file/6_llm_learning.pdf"
loader = PyPDFLoader(file_path=pdf_path)
pages = loader.load()

print(pages[0].page_content[0:500])

面向开发者的  LLM 入门课程  
简介  
LLM 正在逐步改变人们的生活，而对于开发者，如何基于  LLM 提供的  API 快速、便捷地开发一些具备更 
强能力、集成 LLM 的应用，来便捷地实现一些更新颖、更实用的能力，是一个急需学习的重要能力。 
由吴恩达老师与  OpenAI 合作推出的大模型系列教程，从大模型时代开发者的基础技能出发，深入浅出 
地介绍了如何基于大模型  API 、 LangChain 架构快速开发结合大模型强大能力的应用。其中，《 Prompt 
Engineering for Developers 》教程面向入门  LLM 的开发者，深入浅出地介绍了对于开发者，如何构造  
Prompt 并基于  OpenAI 提供的  API 实现包括总结、推断、转换等多种常用功能，是入门  LLM 开发的经 
典教程；《 Building Systems with the ChatGPT API 》教程面向想要基于  LLM 开发应用程序的开发者， 
简洁有效而又系统全面地介绍了如何基于  ChatGPT API 打造完整的对话系统；《 LangChain fo


#### 1.2 CSV加载器

In [3]:
from langchain_community.document_loaders import CSVLoader

file = "./file/5_city_spot.csv"
loader = CSVLoader(file_path=file)
data = loader.load()

for i in range(10):
    print(data[i].page_content)
    print()

code: 111162
level: 4A
province: 辽宁
city: 本溪
spot: 关门山国家森林公园

code: 111194
level: 3A
province: 天津
city: 天津
spot: 世博华明馆

code: 111485
level: 3A
province: 四川
city: 乐山
spot: 犍为文庙

code: 111911
level: 3A
province: 四川
city: 乐山
spot: 东方佛都

code: 112427
level: 4A
province: 江苏
city: 苏州
spot: 东山民居(明善堂

code: 112514
level: 3A
province: 上海
city: 长宁
spot: 书院人家

code: 112826
level: 4A
province: 江西
city: 赣州
spot: 通天寨

code: 113444
level: 5A
province: 江苏
city: 无锡
spot: 太湖鼋头渚风景区

code: 114461
level: 5A
province: 黑龙江
city: 哈尔滨
spot: 太阳岛风景区

code: 114683
level: 4A
province: 贵州
city: 遵义
spot: 赤水古城垣



#### 1.3 网页加载器

In [4]:
from langchain_community.document_loaders import WebBaseLoader

url = "https://www.baidu.com"
header = {'User-Agent': 'python-requests/2.27.1',
          'Accept-Encoding': 'gzip, deflate, br',
          'Accept': '*/*',
          'Connection': 'keep-alive'}
loader = WebBaseLoader(web_path=url, header_template=header)

pages = loader.load()
print(pages[0].page_content[0:5000])

USER_AGENT environment variable not set, consider setting it to identify your requests.



 百度一下，你就知道                     新闻 hao123 地图 视频 贴吧  登录   更多产品       关于百度 About Baidu  ©2017 Baidu 使用百度前必读  意见反馈 京ICP证030173号        



#### 1.4 数据库加载器
(这里以Mysql为例，本表格SQL文件在项目目录file文件下的 6_sport.sql文件，可自行导入本地数据库进行测试)。

In [4]:
from langchain_community.utilities import SQLDatabase
from langchain_community.document_loaders import SQLDatabaseLoader
from sqlalchemy import create_engine
from dotenv import load_dotenv, find_dotenv
import os

def load_deepseek_config():
    """安全加载DeepSeek API配置
    自动查找并加载项目中的.env文件，保护敏感配置信息
    """
    load_dotenv(find_dotenv())

load_deepseek_config() # 初始化环境变量

# 创建数据库引擎
database = "weather" # 数据库名称
db_uri = os.environ["mysql_conn_str"] + database
engine = create_engine(db_uri)

# 初始化加载器
loader = SQLDatabaseLoader(
    db=SQLDatabase(engine=engine),
    query="SELECT * FROM city_jingdian WHERE city = '南京'",
)

pages = loader.load()
print(pages[0].page_content[0:5000])

OperationalError: (pymysql.err.OperationalError) (1049, "Unknown database 'weather'")
(Background on this error at: https://sqlalche.me/e/20/e3q8)

### 2. 文本分割(Splitter)

文本分割是将长文档拆分为适合处理的短片段的关键步骤。合理的分割策略应考虑：

- **分割粒度**：通常500-1000字符为一个块
- **重叠区域**：保留约20%的重叠以避免信息割裂
- **语义完整性**：尽量在段落或句子边界处分割

常用的分割方法：
- 递归字符文本分割(按字符递归尝试分割)
- 字符文本分割(属于"固定分隔符分割")
- Token文本分割(基于token的精确长度控制，简单但可能破坏语义，且中文不太适配)

为了测试文本分割功能效果，我们先创建一个通用的文本内容：

In [5]:
pdf_path = "./file/6_llm_learning.pdf"
loader = PyPDFLoader(file_path=pdf_path)
pages = loader.load()

content = pages[0].page_content

chunk_size = 200 # 设置块大小
chunk_overlap = 40 # 设置块重叠大小

def show_split_res(text, splits):
    print(f"原始文本长度: {len(text)}字符")
    print(f"分割后块数: {len(splits)}")
    print("\n分割结果预览:")
    for i, chunk in enumerate(splits, 1):
        print(f"\n【块{i}】")
        print(chunk)

#### 2.1 递归字符文本分割 (RecursiveCharacterTextSplitter)

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", " ", ""]
)
split_res = r_splitter.split_text(text=content)
show_split_res(content, split_res)

原始文本长度: 1631字符
分割后块数: 10

分割结果预览:

【块1】
面向开发者的  LLM 入门课程  
简介  
LLM 正在逐步改变人们的生活，而对于开发者，如何基于  LLM 提供的  API 快速、便捷地开发一些具备更 
强能力、集成 LLM 的应用，来便捷地实现一些更新颖、更实用的能力，是一个急需学习的重要能力。 
由吴恩达老师与  OpenAI 合作推出的大模型系列教程，从大模型时代开发者的基础技能出发，深入浅出

【块2】
地介绍了如何基于大模型  API 、 LangChain 架构快速开发结合大模型强大能力的应用。其中，《 Prompt 
Engineering for Developers 》教程面向入门  LLM 的开发者，深入浅出地介绍了对于开发者，如何构造  
Prompt 并基于  OpenAI 提供的  API 实现包括总结、推断、转换等多种常用功能，是入门  LLM 开发的经

【块3】
典教程；《 Building Systems with the ChatGPT API 》教程面向想要基于  LLM 开发应用程序的开发者， 
简洁有效而又系统全面地介绍了如何基于  ChatGPT API 打造完整的对话系统；《 LangChain for LLM

【块4】
Application Development 》教程结合经典大模型开源框架  LangChain ，介绍了如何基于  LangChain 框 
架开发具备实用功能、能力全面的应用程序，《 LangChain Chat With Your Data 》教程则在此基础上进 
一步介绍了如何使用  LangChain 架构结合个人私有数据开发个性化大模型应用。

【块5】
上述教程非常适用于开发者学习以开启基于  LLM 实际搭建应用程序之路。因此，我们将该系列课程翻译 
为中文，并复现其范例代码，也为其中一个视频增加了中文字幕，支持国内中文学习者直接使用，以帮 
助中文学习者更好地学习  LLM 开发；我们也同时实现了效果大致相当的中文  Prompt ，支持学习者感受

【块6】
中文语境下  LLM 的学习使用，对比掌握多语言语境下的  Prompt 设计与  LLM 开发。未来，我们也将加 
入更多  Prompt 高级技巧，以丰富本课程内容，帮助开发者掌握更

#### 2.2 字符文本分割 (CharacterTextSplitter)

In [5]:
from langchain.text_splitter import CharacterTextSplitter

c_splitter = CharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separator= '，'
)
split_res = c_splitter.split_text(text=content)
show_split_res(content, split_res)

Created a chunk of size 223, which is longer than the specified 200


原始文本长度: 1631字符
分割后块数: 9

分割结果预览:

【块1】
面向开发者的  LLM 入门课程  
简介  
LLM 正在逐步改变人们的生活，而对于开发者，如何基于  LLM 提供的  API 快速、便捷地开发一些具备更 
强能力、集成 LLM 的应用，来便捷地实现一些更新颖、更实用的能力，是一个急需学习的重要能力。 
由吴恩达老师与  OpenAI 合作推出的大模型系列教程，从大模型时代开发者的基础技能出发

【块2】
从大模型时代开发者的基础技能出发，深入浅出 
地介绍了如何基于大模型  API 、 LangChain 架构快速开发结合大模型强大能力的应用。其中，《 Prompt 
Engineering for Developers 》教程面向入门  LLM 的开发者，深入浅出地介绍了对于开发者，如何构造  
Prompt 并基于  OpenAI 提供的  API 实现包括总结、推断、转换等多种常用功能

【块3】
是入门  LLM 开发的经 
典教程；《 Building Systems with the ChatGPT API 》教程面向想要基于  LLM 开发应用程序的开发者， 
简洁有效而又系统全面地介绍了如何基于  ChatGPT API 打造完整的对话系统；《 LangChain for LLM 
Application Development 》教程结合经典大模型开源框架  LangChain

【块4】
介绍了如何基于  LangChain 框 
架开发具备实用功能、能力全面的应用程序，《 LangChain Chat With Your Data 》教程则在此基础上进 
一步介绍了如何使用  LangChain 架构结合个人私有数据开发个性化大模型应用。 
上述教程非常适用于开发者学习以开启基于  LLM 实际搭建应用程序之路。因此，我们将该系列课程翻译 
为中文，并复现其范例代码

【块5】
我们将该系列课程翻译 
为中文，并复现其范例代码，也为其中一个视频增加了中文字幕，支持国内中文学习者直接使用，以帮 
助中文学习者更好地学习  LLM 开发；我们也同时实现了效果大致相当的中文  Prompt ，支持学习者感受 
中文语境下  LLM 的学习使用，对比掌握多语言语境下的  Prompt 设计与  LLM 开发。未来，我们

#### 2.3 Token文本分割 (TokenTextSplitter)

In [6]:
from langchain.text_splitter import TokenTextSplitter

t_splitter = TokenTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)
split_res = t_splitter.split_text(text=content)
show_split_res(content, split_res)

原始文本长度: 1631字符
分割后块数: 13

分割结果预览:

【块1】
面向开发者的  LLM 入门课程  
简介  
LLM 正在逐步改变人们的生活，而对于开发者，如何基于  LLM 提供的  API 快速、便捷地开发一些具备更 
强能力、集成 LLM 的应用，来便捷地实现一些更新颖、更实用的能力，是一个急�

【块2】
实现一些更新颖、更实用的能力，是一个急需学习的重要能力。 
由吴恩达老师与  OpenAI 合作推出的大模型系列教程，从大模型时代开发者的基础技能出发，深入浅出 
地介绍了如何基于大模型  API 、 LangChain 架构快速开发结

【块3】
了如何基于大模型  API 、 LangChain 架构快速开发结合大模型强大能力的应用。其中，《 Prompt 
Engineering for Developers 》教程面向入门  LLM 的开发者，深入浅出地介绍了对于开发者，如何构造  
Prompt 并基于  OpenAI 提供的  API 实现包括总结、推断、转�

【块4】
  OpenAI 提供的  API 实现包括总结、推断、转换等多种常用功能，是入门  LLM 开发的经 
典教程；《 Building Systems with the ChatGPT API 》教程面向想要基于  LLM 开发应用程序的开发者， 
简洁有效而又系统全面地介绍了如何基于  ChatGPT API 打

【块5】
�又系统全面地介绍了如何基于  ChatGPT API 打造完整的对话系统；《 LangChain for LLM 
Application Development 》教程结合经典大模型开源框架  LangChain ，介绍了如何基于  LangChain 框 
架开发具备实用功能、能力全面的应用程序，《 LangChain Chat With Your Data 》教程则在�

【块6】
�全面的应用程序，《 LangChain Chat With Your Data 》教程则在此基础上进 
一步介绍了如何使用  LangChain 架构结合个人私有数据开发个性化大模型应用。 
上述教程非常适用于开发者学习以开启基于  LLM 实际搭建应用程序之路。因此，我们将�

【块7】
M 实际搭建应用程序之路。因此，我们将该系列课程翻译 


### 三种分割方法对比表：
| 方法                | 代码中的类                     | 保持语义 | 中文支持 | 适用场景               |
|---------------------|-------------------------------|----------|----------|-----------------------|
| 递归字符分割        | RecursiveCharacterTextSplitter | 较好     | ✅       | 通用文本              |
| 固定分隔符分割      | CharacterTextSplitter          | 差       | ✅       | 结构化数据            |
| Token计数分割       | TokenTextSplitter              | 中等     | ❌       | 需要精确token控制的场景|

#### 2.4 针对中文的分割处理建议

In [7]:
# 中文优化的递归分割器
r_splitter_zh = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "。", "！", "？", "\n", "，", "、", ""]
)
split_res_zh = r_splitter_zh.split_text(text=content)
show_split_res(content, split_res_zh)

原始文本长度: 1631字符
分割后块数: 12

分割结果预览:

【块1】
面向开发者的  LLM 入门课程  
简介  
LLM 正在逐步改变人们的生活，而对于开发者，如何基于  LLM 提供的  API 快速、便捷地开发一些具备更 
强能力、集成 LLM 的应用，来便捷地实现一些更新颖、更实用的能力，是一个急需学习的重要能力

【块2】
。 
由吴恩达老师与  OpenAI 合作推出的大模型系列教程，从大模型时代开发者的基础技能出发，深入浅出 
地介绍了如何基于大模型  API 、 LangChain 架构快速开发结合大模型强大能力的应用

【块3】
。其中，《 Prompt 
Engineering for Developers 》教程面向入门  LLM 的开发者，深入浅出地介绍了对于开发者，如何构造  
Prompt 并基于  OpenAI 提供的  API 实现包括总结、推断、转换等多种常用功能，是入门  LLM 开发的经

【块4】
典教程；《 Building Systems with the ChatGPT API 》教程面向想要基于  LLM 开发应用程序的开发者， 
简洁有效而又系统全面地介绍了如何基于  ChatGPT API 打造完整的对话系统；《 LangChain for LLM

【块5】
Application Development 》教程结合经典大模型开源框架  LangChain ，介绍了如何基于  LangChain 框 
架开发具备实用功能、能力全面的应用程序，《 LangChain Chat With Your Data 》教程则在此基础上进 
一步介绍了如何使用  LangChain 架构结合个人私有数据开发个性化大模型应用

【块6】
。 
上述教程非常适用于开发者学习以开启基于  LLM 实际搭建应用程序之路

【块7】
。因此，我们将该系列课程翻译 
为中文，并复现其范例代码，也为其中一个视频增加了中文字幕，支持国内中文学习者直接使用，以帮 
助中文学习者更好地学习  LLM 开发；我们也同时实现了效果大致相当的中文  Prompt ，支持学习者感受 
中文语境下  LLM 的学习使用，对比掌握多语言语境下的  Prompt 设计与  LLM 开发

【块8】
。未来，我们也将加 
入更多  Prompt 高级

### 3. 向量存储(Vectorstore)

向量存储是RAG系统的知识库核心，它将文本转换为向量并建立高效索引。关键考量包括：

- **嵌入模型选择**：
  - 多语言支持
  - 语义表征能力
  - 计算效率

- **向量数据库选型**：
  - FAISS：Facebook开源的轻量级方案
  - Chroma：易于使用的嵌入式数据库
  - Pinecone：全托管的专业服务

考虑到 Chroma 作为向量数据库的 轻量性及其API的简洁性，本案例使用 Chroma 作为向量数据库

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
import shutil

# 每次运行前先清空旧数据
if os.path.exists("./file/6_chroma"):
    shutil.rmtree("./file/6_chroma")
os.makedirs("./file/6_chroma", exist_ok=True)

splits = r_splitter_zh.split_documents(pages)

persist_directory = "./file/6_chroma"  # 指定一个持久化路径
embedding = HuggingFaceEmbeddings(model_name="BAAI/bge-small-zh-v1.5")  # 中文优化模型
vectordb = Chroma.from_documents(
    documents=splits,
    embedding=embedding,
    persist_directory=persist_directory  # 允许我们将persist_directory目录保存到磁盘上
)

NameError: name 'r_splitter_zh' is not defined

### 4. 检索器(Retriever)

检索器负责从知识库中找出相关文档片段。检索技术包括：

- **基础相似度检索器**：纯向量空间最近邻搜索
- **最大边际相关性检索器**：兼顾相关性与多样性
- **自查询检索器**：自动解析查询中的元数据过滤条件
- **上下文压缩检索器**：提升结果精确度

#### 4.1 基础相似度检索器 (Similarity Retriever)

In [3]:
question = "告诉我获取并配置OpenAI API key的方法"
print(vectordb.similarity_search(query=question, k=5))

NameError: name 'vectordb' is not defined

#### 4.2 最大边际相关性检索器 (MMR Retriever)
(不仅考虑问题于文本的相关性，还考虑返回的文本之间的相关性，避免返回过多相同文本)

In [ ]:
print(vectordb.max_marginal_relevance_search(query=question, k=5, fetch_k=3))

#### 4.3 自查询检索器 (Self-Query Retriever)

使用语言模型自动解析语句语义,提取过滤信息,大幅降低构建针对性问答系统的难度。

In [ ]:
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.retrievers.self_query.base import SelfQueryRetriever
from dotenv import load_dotenv, find_dotenv
from langchain_deepseek import ChatDeepSeek

def load_deepseek_config():
    """安全加载DeepSeek API配置"""
    # 自动查找并加载.env文件
    load_dotenv(find_dotenv())

load_deepseek_config()

llm = ChatDeepSeek(
    model="deepseek-chat",
    max_retries=2
)

data_field_info = [
    AttributeInfo(
        name="page",
        description="The page from the lecture",
        type="integer"
    ),
]

document_contents = "面向开发者的 LLM 入门课程"

retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vectordb,
    document_contents=question,
    metadata_field_info=data_field_info,
    verbose=True
)

question_with_target_page = "请从第6页中，告诉我获取并配置OpenAI API key的方法"

docs = retriever.invoke(question_with_target_page)
for doc in docs:
    print("文档内容", doc.page_content)
    print()
    print("文档source信息", doc.metadata)
    print()

#### 4.4 上下文压缩检索器 (Contextual Compression Retriever)

所谓压缩是指仅从文档块中提取出和用户问题相关的内容，并舍弃掉那些不相关的

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

def pretty_print_docs(docs):
    print(f"\n{'-' * 100}\n".join([f"Document {i + 1}:\n\n" + d.page_content for i, d in enumerate(docs)]))

# 对源文档进行压缩
compressor = LLMChainExtractor.from_llm(llm)  # 压缩器
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=vectordb.as_retriever()
)

compressed_docs= compression_retriever.invoke(question)
pretty_print_docs(compressed_docs)

### 5. 问答链(QA Chain)

根据上述检索结果，可以用作文档输入来构建我们的问答链。问答链检索技术包括：

- **简单检索问答链**：实现简单直接，但可能丢失分散在多文档的信息
- **带格式控制的问答链**：可以控制回答风格和结构，并提升回答一致性
- **MapReduce问答链**：对每个文档片段独立生成回答，汇总所有中间回答生成最终结果。适合超长文档处理以及需要并行处理的场景。
- **Refine问答链**：类似RNN的序列处理方式，逐步积累上下文信息，解决跨文档信息关联问题。

#### 5.1 简单检索问答

In [ ]:
from langchain.chains import RetrievalQA

def simple_qa_chain(llm, vectordb, question):
    """基础检索问答实现"""
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=vectordb.as_retriever(search_kwargs={"k": 3})  # 返回top3相关文档
    )
    result = qa_chain.invoke({"query": question})
    return result["result"]

# 测试示例
print("=== 简单检索问答 ===")
print(simple_qa_chain(llm, vectordb, question))

#### 5.2 模板化检索问答
通过提示模板控制回答格式和质量：

In [ ]:
from langchain_core.prompts import PromptTemplate

def template_qa_chain(llm, vectordb, question):
    """带格式控制的问答实现"""
    template = """请根据以下上下文回答问题。要求：
    - 仅使用提供的上下文
    - 回答不超过3句话
    - 以"回答："开头
    - 结尾添加"数据来源：相关技术文档"

    上下文：{context}
    问题：{question}
    回答："""

    QA_PROMPT = PromptTemplate.from_template(template)

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=vectordb.as_retriever(),
        chain_type_kwargs={"prompt": QA_PROMPT},
        return_source_documents=True
    )
    result = qa_chain.invoke({"query": question})
    return result["result"]

# 测试示例
print("\n=== 模板化问答 ===")
print(template_qa_chain(llm, vectordb, question))

#### 5.3 MapReduce问答链
对每个文档片段独立生成回答，汇总所有中间回答生成最终结果

In [ ]:
def refine_qa(llm, vectordb, question):
    """渐进式精炼的问答链""" 
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=vectordb.as_retriever(),
        chain_type="refine",
        verbose=True
    )
    result = qa_chain.invoke({"query": question})
    return result["result"]

print("\n=== Refine问答 ===")
print(refine_qa(llm, vectordb, question))

#### 5.4 Refine问答链
渐进式精炼的问答链，解决信息分散问题，类似RNN的序列处理方式，逐步积累上下文信息。

In [ ]:
def refine_qa(llm, vectordb, question):
    """渐进式精炼的问答链""" 
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=vectordb.as_retriever(),
        chain_type="refine",
        verbose=True
    )
    result = qa_chain.invoke({"query": question})
    return result["result"]

print("\n=== Refine问答 ===")
print(refine_qa(llm, vectordb, question))

#### 5.5 技术对比
| 方法          | 优点                  | 缺点                  | 适用场景              |
|---------------|-----------------------|-----------------------|-----------------------|
| 简单检索      | 响应快，实现简单      | 信息整合能力弱        | 简单问答，信息集中    |
| 模板化        | 回答格式可控          | 灵活性较低            | 需要标准化输出的场景  |
| MapReduce     | 处理长文档能力强      | 速度慢，成本高        | 超长文档分析          |
| Refine        | 信息整合效果好        | 实现复杂度高          | 跨文档信息关联        |